# KAIROS: NIFTY 50 Market Movement Analysis
UML501 — Machine Learning

Data acquisition, preprocessing, exploratory analysis, candidate predictors and classification targets.

**Execution:** Place the two source CSV files in the working directory or set `KAIROS_DATA_DIR` to their location. The optional acquisition function is not executed by default.

**Development assistance:** AI-assisted coding and editing were used in preparing this notebook; the methodology and results should be independently reviewed.

## 1. Problem statement
Investigate whether historical NIFTY 50 prices and India VIX can predict significant net price movements over the next five trading sessions. Forecasts are issued after market close. The provisional threshold is ±3%.

## 2. Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import os

## 3. Data collection
### 3.1 Source and collection method
Historical NIFTY 50 observations were obtained from NSE Indices. The optional acquisition function below reconstructs the original batched HTTP request procedure; the live endpoint has not been retested. Fixed CSV snapshots are used for reproducibility. The original India VIX acquisition script is unavailable.
### 3.2 Optional NIFTY acquisition code

In [ ]:
def collect_nifty_history(start_date, end_date):
    import requests
    import json
    import time
    from datetime import timedelta

    url = 'https://www.niftyindices.com/BackPage/getHistoricaldatatabletoString'
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Content-Type': 'application/json',
        'X-Requested-With': 'XMLHttpRequest',
        'Referer': 'https://www.niftyindices.com/reports/historical-data',
        'Origin': 'https://www.niftyindices.com'
    }
    if start_date > end_date:
        raise ValueError('Start date must precede end date.')
    frames = []
    with requests.Session() as session:
        session.headers.update(headers)
        current_start = start_date
        while current_start <= end_date:
            current_end = min(current_start + timedelta(days=364), end_date)
            cinfo = ("{'name':'NIFTY 50','startDate':'" + current_start.strftime('%m/%d/%Y')
                     + "','endDate':'" + current_end.strftime('%m/%d/%Y')
                     + "','indexName':'NIFTY 50'}")
            for attempt in range(3):
                try:
                    response = session.post(url, json={'cinfo': cinfo}, timeout=90)
                    response.raise_for_status()
                    result = response.json()
                    if isinstance(result, dict):
                        result = result['d']
                    if isinstance(result, str):
                        result = json.loads(result)
                    if not isinstance(result, list):
                        raise ValueError('Unexpected response format')
                    if result:
                        frames.append(pd.DataFrame(result))
                    elif (current_end - current_start).days >= 7:
                        raise ValueError('Unexpected empty multi-day batch')
                    break
                except (requests.RequestException, ValueError, KeyError) as exc:
                    if attempt == 2:
                        raise RuntimeError(f'Failed batch {current_start} to {current_end}') from exc
                    time.sleep(2 ** attempt)
            current_start = current_end + timedelta(days=1)
            time.sleep(1.5)
    if not frames:
        raise ValueError('No observations returned')
    result = pd.concat(frames, ignore_index=True).rename(columns={
        'HistoricalDate': 'Date', 'date': 'Date', 'OPEN': 'Open',
        'HIGH': 'High', 'LOW': 'Low', 'CLOSE': 'Close'
    })[['Date', 'Open', 'High', 'Low', 'Close']]
    result['Date'] = pd.to_datetime(result['Date'], errors='raise')
    for col in ['Open', 'High', 'Low', 'Close']:
        result[col] = pd.to_numeric(result[col], errors='raise')
    return result.sort_values('Date').reset_index(drop=True)

# Optional historical request (not run):
# from datetime import date
# collected = collect_nifty_history(date(2006, 11, 1), date(2026, 11, 14))
# collected.to_csv('NIFTY_RECOLLECTED.csv', index=False)

### 3.3 Load source datasets
The source CSV files remain unchanged throughout the analysis.

In [ ]:
base = Path(os.environ.get('KAIROS_DATA_DIR', str(Path.cwd())))
nifty_file = base / 'KAIROS_NIFTY50_NSE_2006_2026.csv'
vix_file = base / 'KAIROS_INDIA_VIX_2008_2026.csv'
for file in (nifty_file, vix_file):
    if not file.is_file():
        raise FileNotFoundError(f'Upload {file.name} to {base} before continuing.')
nifty = pd.read_csv(nifty_file)
vix = pd.read_csv(vix_file)
print('NIFTY:', nifty.shape)
print('VIX:', vix.shape)
display(nifty.head(), vix.head())

## 4. Data preprocessing
### 4.1 Check column names and data types

In [ ]:
prices = ['Open', 'High', 'Low', 'Close']
assert set(nifty.columns) == set(['Date'] + prices)
assert set(vix.columns) == {'Date', 'VIX_Close'}
print(nifty.dtypes)
print(vix.dtypes)

### 4.2 Convert dates and sort chronologically

In [ ]:
for frame in (nifty, vix):
    frame['Date'] = pd.to_datetime(frame['Date'], errors='raise')
    frame.sort_values('Date', inplace=True)
    frame.reset_index(drop=True, inplace=True)
    assert frame.Date.notna().all()
    assert frame.Date.is_monotonic_increasing
print('NIFTY:', nifty.Date.min().date(), 'to', nifty.Date.max().date())
print('VIX:', vix.Date.min().date(), 'to', vix.Date.max().date())

### 4.3 Check missing values

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'missing values:')
    print(frame.isna().sum())
    assert frame.notna().all().all()

### 4.4 Check duplicate dates

In [ ]:
for name, frame in [('NIFTY', nifty), ('VIX', vix)]:
    print(name, 'duplicate dates:', frame.Date.duplicated().sum())
    assert frame.Date.is_unique

### 4.5 Validate OHLC and VIX values

In [ ]:
assert np.isfinite(nifty[prices]).all().all()
assert (nifty[prices] > 0).all().all()
assert (nifty.High >= nifty[['Open', 'Close', 'Low']].max(axis=1)).all()
assert (nifty.Low <= nifty[['Open', 'Close', 'High']].min(axis=1)).all()
assert np.isfinite(vix.VIX_Close).all() and (vix.VIX_Close > 0).all()
print('OHLC and VIX checks passed.')

### 4.6 Align the date ranges
Use the common observation period. Earlier NIFTY records are excluded from the joint experiment.

In [ ]:
start = max(nifty.Date.min(), vix.Date.min())
end = min(nifty.Date.max(), vix.Date.max())
assert start <= end
aligned = nifty.loc[nifty.Date.between(start, end)].copy()
vix_aligned = vix.loc[vix.Date.between(start, end)].copy()
assert vix_aligned.Date.isin(aligned.Date).all()
print('Common period:', start.date(), 'to', end.date())
print('Excluded NIFTY rows:', len(nifty) - len(aligned))
print('Aligned NIFTY rows:', len(aligned))
print('Aligned VIX rows:', len(vix_aligned))

### 4.7 Merge the datasets
Use a one-to-one left join to preserve aligned NIFTY sessions.

In [ ]:
df = aligned.merge(vix_aligned, on='Date', how='left', validate='one_to_one')
assert len(df) == len(aligned)
assert df.Date.is_unique and df.Date.is_monotonic_increasing
print('Merged shape before missing-value flag:', df.shape)
print('Missing VIX observations:', df.VIX_Close.isna().sum())
display(df.loc[df.VIX_Close.isna()])

### 4.8 Handle missing VIX observations
Flag missing readings before carrying forward the last available earlier value. A carried-forward reading is not an observed value.

In [ ]:
df['VIX_missing'] = df.VIX_Close.isna().astype(int)
print('Missing VIX dates:', df.loc[df.VIX_missing.eq(1), 'Date'].dt.date.tolist())
df['VIX_Close'] = df.VIX_Close.ffill()
assert df.notna().all().all()
assert len(df) == len(aligned)
print('Remaining missing values:', df.isna().sum().sum())
print('Cleaned dataset shape:', df.shape)
display(df.head())

### 4.9 Export the cleaned combined dataset

In [ ]:
output_path = base / 'KAIROS_COMBINED_CLEANED.csv'
df.to_csv(output_path, index=False)
print('Combined dataset saved:', output_path.name, '| rows:', len(df))
# from google.colab import files
# files.download(str(output_path))

## 5. Chronological experiment boundaries
Use 70% for training, 15% for validation and 15% for testing. EDA uses the training period only. The test partition is a procedural holdout rather than an independent external dataset because the historical snapshot was previously examined.

In [ ]:
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)
train = df.iloc[:train_end].copy()
for name, part in [('Train', train), ('Validation', df.iloc[train_end:val_end]), ('Test', df.iloc[val_end:])]:
    print(name, len(part), part.Date.min().date(), part.Date.max().date())

## 6. Exploratory Data Analysis (EDA)
### 6.1 Descriptive statistics

In [ ]:
display(train[prices + ['VIX_Close']].describe().round(2))

### 6.2 Market trends
NIFTY closing levels and implied volatility are plotted separately.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Close, linewidth=1)
axes[0].set(title='NIFTY 50 closing level', ylabel='Index points')
axes[1].plot(train.Date, train.VIX_Close, linewidth=1)
axes[1].set(title='India VIX', ylabel='Index level', xlabel='Date')
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### 6.3 Returns and rolling volatility
Compute daily percentage returns and trailing 20-session standard deviation.

In [ ]:
train['Return_1d'] = train.Close.pct_change() * 100
train['Volatility_20d'] = train.Return_1d.rolling(20).std()
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train.Date, train.Return_1d, linewidth=0.7)
axes[0].set(title='Daily NIFTY returns', ylabel='Return (%)')
axes[1].plot(train.Date, train.Volatility_20d, linewidth=1)
axes[1].set(title='20-session rolling volatility', ylabel='Daily std. dev. (%)', xlabel='Date')
fig.tight_layout()
plt.show()
print(train.Return_1d.describe().round(3))
print('Largest daily decline:', train.loc[train.Return_1d.idxmin(), ['Date','Return_1d']].to_dict())
print('Largest daily rise:', train.loc[train.Return_1d.idxmax(), ['Date','Return_1d']].to_dict())

### 6.4 VIX and market returns
Pearson correlation measures linear association, not predictive performance.

In [ ]:
train['VIX_Change'] = train.VIX_Close.pct_change() * 100
train['Abs_Return'] = train.Return_1d.abs()
cols = ['Return_1d', 'Abs_Return', 'VIX_Close', 'VIX_Change', 'Volatility_20d']
corr = train[cols].corr()
display(corr.round(3))
plt.figure(figsize=(7, 4))
plt.scatter(train.VIX_Change, train.Return_1d, s=8, alpha=0.35)
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.xlabel('Daily VIX change (%)')
plt.ylabel('Daily NIFTY return (%)')
plt.title('Same-session return and VIX change')
plt.tight_layout()
plt.show()

### 6.5 Candidate target distribution
Use a provisional five-session return threshold of ±3%. Exclude unknown outcomes.

In [ ]:
forward = (train.Close.shift(-5) / train.Close - 1) * 100
observed = forward.dropna()
labels = pd.Series(np.select([observed < -3, observed > 3], ['Major Down', 'Major Up'], default='Ordinary'), index=observed.index)
order = ['Major Down', 'Ordinary', 'Major Up']
counts = labels.value_counts().reindex(order, fill_value=0)
summary = pd.DataFrame({'Count': counts, 'Percent': counts / len(labels) * 100})
display(summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed, bins=60, edgecolor='white')
for boundary in (-3, 3):
    axes[0].axvline(boundary, linestyle='--', linewidth=1.4)
axes[0].set(title='Five-session forward returns', xlabel='Return (%)', ylabel='Observations')
axes[1].bar(order, counts)
axes[1].set(title='Candidate target distribution', ylabel='Observations')
fig.tight_layout()
plt.show()
assert len(labels) == len(train) - 5
print('Excluded rows without a within-training outcome:', len(train) - len(labels))

### 6.6 EDA conclusions
Class imbalance requires macro F1 and per-class precision and recall alongside accuracy. Same-session correlations are not evidence of future forecasting skill.

## 7. Feature engineering
### 7.1 Candidate predictors
All features use information known at market close. Trailing indicators may use earlier history across partition boundaries.

In [ ]:
def make_features(data):
    features = pd.DataFrame(index=data.index)
    close = data['Close']
    vix_close = data['VIX_Close']
    daily_return = close.pct_change(fill_method=None) * 100
    features['Return_1d'] = daily_return
    features['Return_5d'] = close.pct_change(5, fill_method=None) * 100
    features['Volatility_20d'] = daily_return.rolling(20).std()
    features['MA_20_gap_pct'] = (close / close.rolling(20).mean() - 1) * 100
    features['Intraday_range_pct'] = (data['High'] - data['Low']) / data['Open'] * 100
    features['VIX_Close'] = vix_close
    features['VIX_change_1d'] = vix_close.pct_change(fill_method=None) * 100
    features['VIX_missing'] = data['VIX_missing']
    # Mark changes involving an imputed VIX as missing rather than inventing a move.
    unreliable = data['VIX_missing'].eq(1) | data['VIX_missing'].shift(1, fill_value=0).eq(1)
    features.loc[unreliable, 'VIX_change_1d'] = np.nan
    return features

X = make_features(df)
X_train = X.iloc[:train_end].copy()
print('Candidate predictors:', X_train.shape[1])
print('Training rows with all predictors available:', X_train.dropna().shape[0])
display(X_train.iloc[20:25].round(3))

### 7.2 Feature validation and leakage checks
Prefix invariance tests for accidental use of future observations.

In [ ]:
for cutoff in (50, 250, train_end, val_end):
    pd.testing.assert_frame_equal(make_features(df.iloc[:cutoff]), X.iloc[:cutoff])
assert not np.isinf(X.to_numpy(dtype=float)).any()
assert X.index.equals(df.index)
print('Prefix-invariance, finite-value and alignment checks passed.')
print('Training missing counts:')
print(X_train.isna().sum())

## 8. Target variable creation
### 8.1 Five-session classification labels
Returns above +3% map to 1, below −3% to −1, and the inclusive interval to 0. A numerical tolerance protects threshold boundaries. Outcomes are generated separately in each partition.

In [ ]:
"""Five-session classification labels, with partition boundary protection."""
import numpy as np
import pandas as pd


def make_targets(data, horizon=5, threshold=3.0):
    if horizon < 1 or threshold <= 0:
        raise ValueError('horizon and threshold must be positive')
    if not data.Date.is_monotonic_increasing or not data.Date.is_unique:
        raise ValueError('Dates must be unique and sorted')
    forward = (data.Close.shift(-horizon) / data.Close - 1) * 100
    boundary = np.isclose(forward.abs(), threshold, rtol=0, atol=1e-10)
    target = pd.Series(pd.NA, index=data.index, dtype='Int64')
    target.loc[forward.notna()] = 0
    target.loc[(forward > threshold) & ~boundary] = 1
    target.loc[(forward < -threshold) & ~boundary] = -1
    return pd.DataFrame({'Outcome_date': data.Date.shift(-horizon),
                         'Forward_return_5d_pct': forward, 'Target': target})


### 8.2 Prepare aligned training observations
Forward returns and outcome dates are retained for evaluation only. Exclude unavailable features and unknown outcomes.

In [ ]:
partitions = {'train': slice(0, train_end), 'validation': slice(train_end, val_end), 'test': slice(val_end, len(df))}
feature_columns = X.columns.tolist()
training_labels = make_targets(df.iloc[partitions['train']])
training_mask = X.iloc[partitions['train']].notna().all(axis=1) & training_labels.Target.notna()
X_fit = X.iloc[partitions['train']].loc[training_mask, feature_columns].copy()
y_fit = training_labels.loc[training_mask, 'Target'].astype(int)
print('Eligible training observations:', len(y_fit))
print('Excluded for unknown outcomes:', training_labels.Target.isna().sum())
print('Excluded for unavailable predictors among labelled rows:', ((~X.iloc[partitions['train']].notna().all(axis=1)) & training_labels.Target.notna()).sum())
counts_fit = y_fit.value_counts().reindex([-1, 0, 1], fill_value=0)
print(pd.DataFrame({'Count': counts_fit, 'Percent': (counts_fit / len(y_fit) * 100).round(2)}))

### 8.3 Validate target boundaries and alignment

In [ ]:
for future, expected in [(96, -1), (97, 0), (100, 0), (103, 0), (104, 1)]:
    example = pd.DataFrame({'Date': pd.date_range('2020-01-01', periods=6), 'Close': [100]*5 + [future]})
    check = make_targets(example)
    assert check.Target.iloc[0] == expected
    assert check.Target.iloc[1:].isna().all()
for part in partitions.values():
    block = df.iloc[part]
    result = make_targets(block)
    assert result.Target.tail(5).isna().all()
    assert result.Outcome_date.dropna().le(block.Date.max()).all()
    assert result.Target.notna().sum() == len(block) - 5
assert X_fit.index.equals(y_fit.index)
assert not {'Target', 'Outcome_date', 'Forward_return_5d_pct'}.intersection(X_fit.columns)
assert X_fit.notna().all().all()
print('Target threshold, partition boundary and feature alignment checks passed.')

## 9. Scope and limitations
The feature set and target definition are provisional pending chronological validation. A five-session separation is required at cross-validation boundaries. No predictive performance is reported at this stage. Fixed historical snapshots are used; live collection has not been retested.